In [1]:
import torch
import numpy as np
from lime.lime_text import LimeTextExplainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [3]:
# load tokenizer directly from HuggingFace because of version mismatch
lime_tokenizer = AutoTokenizer.from_pretrained('roberta-base')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
# load model from saved folder
lime_model = AutoModelForSequenceClassification.from_pretrained('models/roberta_sarcasm')
lime_model = lime_model.to(device)
lime_model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [5]:
explainer = LimeTextExplainer(class_names=['Not Sarcastic', 'Sarcastic'])
print("LIME ready")

LIME ready


In [6]:
def predict_proba(texts):
    encodings = lime_tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors='pt'
    )
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    with torch.no_grad():
        outputs = lime_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

    return probs

In [7]:
# Test run examples 
examples = [
    "Oh great, another Monday. Just what I needed.",
    "Absolute legend, parked his ute right across my driveway. Good onya mate.",
    "Coz we all have free internet.",
    "Traditional friendly pub. Excellent beer.",
]

In [8]:
for i, text in enumerate(examples):
    print(f"\nExample {i+1}: {text}")
    exp = explainer.explain_instance(text, predict_proba, num_features=6, num_samples=100)
    pred_probs = predict_proba([text])
    
    if pred_probs[0][1] > 0.5:
        pred_label = 'Sarcastic'
    else:
        pred_label = 'Not Sarcastic'
    print(f"Prediction: {pred_label} (confidence: {max(pred_probs[0]):.2f})")
    
    print("Top influential words:")
    for word, weight in exp.as_list():
        if weight > 0:
            direction = "→ Sarcastic"
        else:
            direction = "→ Not Sarcastic"
        print(f"'{word}': {weight:.4f} {direction}")


Example 1: Oh great, another Monday. Just what I needed.
Prediction: Not Sarcastic (confidence: 0.99)
Top influential words:
'needed': -0.5387 → Not Sarcastic
'great': -0.1750 → Not Sarcastic
'another': -0.1393 → Not Sarcastic
'Monday': 0.0975 → Sarcastic
'what': -0.0958 → Not Sarcastic
'Just': -0.0819 → Not Sarcastic

Example 2: Absolute legend, parked his ute right across my driveway. Good onya mate.
Prediction: Not Sarcastic (confidence: 1.00)
Top influential words:
'Absolute': -0.3187 → Not Sarcastic
'Good': -0.2058 → Not Sarcastic
'my': 0.1066 → Sarcastic
'right': -0.0692 → Not Sarcastic
'ute': 0.0658 → Sarcastic
'onya': 0.0577 → Sarcastic

Example 3: Coz we all have free internet.
Prediction: Not Sarcastic (confidence: 1.00)
Top influential words:
'internet': -0.2496 → Not Sarcastic
'have': -0.1882 → Not Sarcastic
'we': 0.1064 → Sarcastic
'Coz': 0.0705 → Sarcastic
'free': -0.0347 → Not Sarcastic
'all': -0.0320 → Not Sarcastic

Example 4: Traditional friendly pub. Excellent beer.